# Integración del agente

**Fase:** E13

## Objetivo

Componer el prompt con diagnóstico, confianza, evidencia observacional, atribución y fragmentos recuperados, y obtener salida estructurada.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II reales, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

El LLM redacta y organiza; no diagnostica ni aporta conocimiento propio como fuente.

## Hipótesis

Un modelo de lenguaje pequeño (3B), local, al que se le entrega la decisión ya tomada y la evidencia ya verificada, puede **redactarla respetando el veredicto y citando solo lo que existe**, y un ajuste fino **solo del formato de la tarea** (sin contenido del manual en los pesos) mejora la tasa de salidas válidas sin introducir citas inventadas.

## Datos

- Nueve casos anotados de `configs/goldset_mazda3_bk.yaml` sobre ventanas de **validación** de `synthetic_v2` (el test sigue cerrado).
- Ajuste: 62 ejemplos derivados de ejecuciones auténticas del pipeline (55 entrenamiento, 7 validación), equilibrados ABSTAIN/EXPLAIN, reparto por caso. Ver informe E13.

## Método

1. **Antes:** Qwen2.5-3B-Instruct base, motor `transformers`, MPS, bf16, decodificación determinista.
2. **Ajuste:** LoRA rango 16 sobre `q/k/v/o_proj` (0,24 % de parámetros), 4 épocas, lr 2e-4, longitud 3 072, pérdida solo sobre la completion, semilla 20260901.
3. **Después:** el mismo motor cargando el adaptador. Mismas nueve preguntas, mismo prompt, mismo verificador.

El «antes» **no** es la medición con Ollama: aquella usaba un modelo cuantizado y otro motor, y no sería comparable.

## Métricas

Por caso, calculadas fuera del modelo por `run_agent` y `verify_citations`:

- **Salida válida:** JSON que cumple el esquema y no cita nada inexistente.
- **Respeta el veredicto:** redacta EXPLAIN si el verificador dijo EXPLAIN, ABSTAIN si dijo ABSTAIN.
- **Citas inventadas:** identificadores con forma correcta que no corresponden a nada entregado.
- **Esquema roto:** la salida no valida (y por qué).

## Criterio de éxito o refutación

Fijado antes de la segunda pasada (la primera reveló un fallo de esquema propio, corregido en `e2d7c59`; sus registros se conservan como `esquema_v1`):

- **Éxito:** después ≥ antes en salidas válidas, **cero** citas inventadas en ambos, y el ajustado no respeta el veredicto menos veces que el base.
- **Refutación:** el ajustado introduce citas inventadas, o respeta el veredicto menos que el base. Una mejora en validez que llegue a costa de cualquiera de las dos no cuenta.


In [1]:
# Los notebooks importan del paquete; no duplican lógica científica.
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import mecanico_ia_exp

mecanico_ia_exp.__version__

'0.1.0'

## Ejecución

Las dos mediciones se lanzan desde la CLI y dejan un registro JSON por proveedor. El cuaderno **no** vuelve a generar: lee los registros y los resume con el paquete.

```bash
uv run mecanico-exp finetune-slm --epochs 4
uv run mecanico-exp sufficiency --llm transformers
uv run mecanico-exp sufficiency --llm transformers-lora
```

In [2]:
import json

import pandas as pd

from mecanico_ia_exp.config import paths
from mecanico_ia_exp.llm.report import compare_records, per_case

REGISTROS = {
    "base (esquema v1)": paths.EXPERIMENTS / "E13_agente_transformers_esquema_v1.json",
    "base + LoRA (esquema v1)": paths.EXPERIMENTS / "E13_agente_transformers-lora_esquema_v1.json",
    "base": paths.EXPERIMENTS / "E13_agente_transformers.json",
    "base + LoRA": paths.EXPERIMENTS / "E13_agente_transformers-lora.json",
}
compare_records(REGISTROS)

,proveedor,válidas,respetan veredicto,citas inventadas,esquema roto,n
0,base (esquema v1),5,9,0,4,9
1,base + LoRA (esquema v1),6,9,0,3,9
2,base,7,9,0,2,9
3,base + LoRA,7,9,0,2,9


In [3]:
# Qué aprendió el adaptador, y cuánto: registro escrito por finetune().
historial = json.loads((paths.MODELS / "slm_lora" / "finetune_config.json").read_text())
h = historial["history"]
print(
    f"entrenables: {h['n_trainable_params'] / h['n_total_params']:.2%} · "
    f"{h['seconds'] / 60:.0f} min · recortados {h['n_truncated']}/{h['n_train']}"
)
pd.DataFrame(
    {
        "época": range(1, len(h["train_loss"]) + 1),
        "entrenamiento": h["train_loss"],
        "validación": h["val_loss"],
    }
).round(3)

entrenables: 0.24% · 88 min · recortados 28/55


,época,entrenamiento,validación
0,1,0.480,0.040
1,2,0.058,0.020
2,3,0.031,0.013
3,4,0.027,0.018


In [4]:
# Caso a caso, segunda pasada: dónde falla cada uno.
tabla = per_case({k: v for k, v in REGISTROS.items() if "v1" not in k})
tabla.pivot(index="orden", columns="proveedor", values=["caso", "redactó", "esquema"])

caso  \
proveedor                                  base   
orden                                             
1               F02_maf_drift__highway__01@120s   
2          F03_thermostat_stuck__urban__08@120s   
3                     F04_o2_lag__idle__02@120s   
4                 F05_misfire__highway__09@180s   
5                F01_vacuum_leak__idle__10@120s   
6                F01_vacuum_leak__idle__10@120s   
7                F01_vacuum_leak__idle__10@120s   
8                F01_vacuum_leak__idle__10@120s   
9                F01_vacuum_leak__idle__10@120s   

                                                 redactó             esquema  \
proveedor                           base + LoRA     base base + LoRA    base   
orden                                                                          
1               F02_maf_drift__highway__01@120s  ABSTAIN     ABSTAIN      ok   
2          F03_thermostat_stuck__urban__08@120s        —     EXPLAIN   FALLA   
3                     F04_o2_lag__idle__02@120s  EXPLAIN     EXPLAIN      ok   
4                 F05_misfire__highway__09@180s  ABSTAIN     ABSTAIN      ok   
5                F01_vacuum_leak__idle__10@120s        —     EXPLAIN   FALLA   
6                F01_vacuum_leak__idle__10@120s  ABSTAIN     ABSTAIN      ok   
7                F01_vacuum_leak__idle__10@120s  ABSTAIN           —      ok   
8                F01_vacuum_leak__idle__10@120s  ABSTAIN           —      ok   
9                F01_vacuum_leak__idle__10@120s  ABSTAIN     ABSTAIN      ok   

                       
proveedor base + LoRA  
orden                  
1                  ok  
2                  ok  
3                  ok  
4                  ok  
5                  ok  
6                  ok  
7               FALLA  
8               FALLA  
9                  ok

In [5]:
# Texto literal de los fallos de esquema del ajustado, si los hay.
# El registro lo conserva para no tener que volver a generar.
ruta = REGISTROS["base + LoRA"]
if ruta.is_file():
    for r in json.loads(ruta.read_text())["runs"]:
        if r["schema_error"]:
            print(
                r["case_id"],
                "→",
                r["schema_error"].splitlines()[-2:],
                "\n  ",
                (r.get("raw_text") or "")[:200],
            )

F01_vacuum_leak__idle__10@120s → ["  Input should be 'INSUFFICIENT_EVIDENCE', 'NO_APPLICABLE_DOCUMENTATION', 'CONTRADICTORY_EVIDENCE', 'LOW_MODEL_CONFIDENCE', 'NO_APPLICABLE_DOCUMENTARY_EVIDENCE', 'LOW_RETRIEVAL_SUPPORT', 'TOO_FEW_SUPPORTING_CHUNKS', 'TOO_FEW_DISTINCT_SOURCES', 'LOW_CLAIM_COVERAGE', 'CONTRADICTION_DETECTED' or 'NO_OBSERVATIONAL_EVIDENCE' [type=enum, input_value='LOW_CLAIM_COVERDGE', input_type=str]", '    For further information visit https://errors.pydantic.dev/2.13/v/enum'] 
   {"decision": "ABSTAIN", "reason": "LOW_CLAIM_COVERDGE", "missing_evidence": ["cobertura 0.00 < 0.80"], "notes": "No existe evidencia suficiente para sostener una explicación diagnóstica."}
F01_vacuum_leak__idle__10@120s → ["  Input should be 'INSUFFICIENT_EVIDENCE', 'NO_APPLICABLE_DOCUMENTATION', 'CONTRADICTORY_EVIDENCE', 'LOW_MODEL_CONFIDENCE', 'NO_APPLICABLE_DOCUMENTARY_EVIDENCE', 'LOW_RETRIEVAL_SUPPORT', 'TOO_FEW_SUPPORTING_CHUNKS', 'TOO_FEW_DISTINCT_SOURCES', 'LOW_CLAIM_COVERAGE', 'CONTRAD

## Resultados

Segunda pasada, esquema unificado, mismo motor y mismas nueve preguntas (tabla de arriba, filas «base» y «base + LoRA»):

| | Válidas | Respeta veredicto | Citas inventadas | Esquema roto | Latencia media · máx |
|---|---|---|---|---|---|
| Base | 7/9 | 9/9 | 0 | 2 (C02, C05: JSON roto en EXPLAIN) | 20,4 s · 100,7 s |
| Base + LoRA v1 | 7/9 | 9/9 | 0 | 2 (C07, C08: `LOW_CLAIM_COVERDGE`) | 14,7 s · 38,0 s |

Estructura de los EXPLAIN válidos (observaciones, fragmentos) por afirmación: base `[(1, 3), (5, 2)]` en su único caso válido; ajustado `[(1, 2), (1, 2), (1, 2)]` en los tres.

Primera pasada (filas «esquema v1»): 5/9 → 6/9. Tres de esos fallos eran del esquema, no del modelo.

## Interpretación

- **El criterio se cumple en el límite.** Después = antes en válidas, cero citas inventadas, veredicto respetado 9/9 en ambos. No es una mejora del recuento y no se presenta como tal.
- **Lo que el ajuste apuntaba, lo consiguió.** Una afirmación por observación, con sus fragmentos, sin parafrasear el manual, sin afirmar causa, y en un tercio del tiempo máximo. El base, cuando acierta, cuelga cinco observaciones de una afirmación y escribe «el manual indica que…» como afirmación propia.
- **Lo que rompió tiene causa identificada en los datos**, no en el modelo: el objetivo de entrenamiento traducía el motivo del verificador y el prompt mostraba el literal; y la abstención por cobertura, que es la del experimento central, tiene un ejemplo entre 26.
- **Dos de los cuatro «fallos del base» de la primera pasada eran míos.** El base sube de 5/9 a 7/9 al corregir el esquema sin tocar el modelo. La medición del modelo depende de la corrección del instrumento.

## Limitaciones

- Nueve casos. Las diferencias de una unidad no separan nada; lo que separa es la estructura caso a caso.
- Datos sintéticos calibrados con el manual; la PI habla de datos reales.
- 28 de 55 prompts recortados por la izquierda en el ajuste.
- Siete ejemplos de validación en el ajuste: la subida de la pérdida en la época 4 no es concluyente.
- Decodificación determinista en un solo motor (MPS, bf16). Otro motor daría otro «antes».

## Conclusión experimental

Con la evidencia entregada hecha, un modelo de 3B local **respeta el veredicto y no inventa citas** ni antes ni después del ajuste: eso lo garantiza el diseño (barrera fuera del modelo), no el modelo. El ajuste de formato, con 0,24 % de parámetros y sin contenido del manual en los pesos, **cambia la forma de la explicación** hacia la que se quería y **no mejora el recuento** por un fallo de vocabulario en los datos de ajuste. La iteración 2 corrige ese fallo y solo ese; se mide con las mismas nueve preguntas.

## Artefactos generados

- `models/slm_lora/` — adaptador v1 (binario fuera de Git; `adapter_config.json` y `finetune_config.json` dentro).
- `experiments/E13_agente_transformers.json`, `..._transformers-lora.json` — segunda pasada, con texto crudo y error completo por caso.
- `experiments/E13_agente_*_esquema_v1.json` — primera pasada, conservada como prueba del fallo de esquema.
- `experiments/E13b_training_data.json` — resumen del conjunto de ajuste.
- `docs/experiments/E13_agente_slm.md` — informe.
